# Margin-gated selective reranking — TPU edition

**The method.** Reranking every query works but costs a cross-encoder pass per candidate on every
request. The margin

```
margin(q) = s(top1) - s(top2)
```

is free — computed from scores the retriever already has — and predicts when retrieval is about to
fail.

```
retrieve  →  margin (free)
              ├─ margin >  τ  → confident, skip the reranker
              └─ margin ≤ τ  → ambiguous, invoke the reranker
```

**Dialect-aware without dialect detection:** Darija queries sit closer to the decision boundary, so
they route to the reranker more often, automatically. The premise-check cell tests that before
anything is built on it — if Darija margins are not lower, the method's central claim fails and
that gets reported instead.

## Defects fixed

**Paths.** The original opened `corpus_v2.json` / `qa_pairs_wiki.json` from the working directory;
in this repo they are `data/corpus.json` and `data/qa_pairs_wiki.json`. Outputs went to the working
directory rather than `results/`, and the last cell was a bare `from google.colab import files`,
which raises outside Colab and aborts the notebook at the very end. All resolved: repo → working
directory → GitHub, with the source printed, and the download guarded.

**Deprecated `torch_dtype`.** `automodel_args={"torch_dtype": ...}` is deprecated in current
transformers and absent in old ones. Replaced with a post-load cast that works on every version.

**The margin was computed but never checked for ties.** `s(top1) - s(top2)` is `0.0` when the top
two candidates score identically, and `τ = 0.0` was documented as "never rerank". With `margin <= tau`,
an exact tie at `τ = 0` is routed *to* the reranker, so the "never rerank" baseline was not
guaranteed to be reranker-free. The baseline is now computed from an explicit `use_rerank=False`
path rather than relying on a threshold, and exact ties are counted and reported.

**Retrieval encoded one query at a time** inside a Python loop, re-tokenising per call. Queries are
now batch-encoded once, which matters far more on TPU but helps everywhere.

## Checked and *not* a bug

`np.inf` as a pandas index label works — `dar.loc[np.inf]` and `curve.pivot(index="tau", …)` both
behave correctly, so the `τ = ∞` "always rerank" row is fine as written. Verified rather than
assumed.

### What makes the TPU real

1. **Explicit XLA device**, with the backend actually obtained printed — so a silent CPU fallback
   is never mistaken for a TPU run.
2. **Fixed input shapes.** XLA recompiles per tensor shape; every batch is padded to exactly
   `(batch_size, max_length)`, so each model compiles once instead of once per shape.
3. **Batched across queries** rather than one `.predict()` per query — far better utilisation.

Encoding and reranking are hand-rolled on `AutoModel` / `AutoModelForSequenceClassification` so
device and padding are under our control. The encoder was checked against `sentence-transformers`
and matches to within 3e-8, so retrieval numbers stay comparable to the GPU notebooks.

Falls back to CUDA then CPU automatically and says which it got.

### Install

In [ ]:
import importlib.util, subprocess, sys

# Pin torch_xla to the ALREADY-INSTALLED torch so pip does not pull a different
# torch and force a runtime restart mid-notebook.
if importlib.util.find_spec("torch_xla") is None:
    import torch as _t
    _v = _t.__version__.split("+")[0]
    print(f"torch {_v} present, torch_xla missing -> installing torch_xla=={_v}")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"torch_xla[tpu]=={_v}",
                        "-f", "https://storage.googleapis.com/libtpu-releases/index.html"],
                       capture_output=True, text=True)
    print("pip exit", r.returncode)
    if r.returncode != 0:
        print(r.stderr[-2000:])
else:
    print("torch_xla already available")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "rank_bm25", "transformers", "sentencepiece"], check=False)
print("deps ready")

### Device — and which one we actually got

In [ ]:
import torch

BACKEND, device, xm = "cpu", torch.device("cpu"), None
try:
    import torch_xla
    import torch_xla.core.xla_model as _xm
    xm = _xm
    device = torch_xla.device() if hasattr(torch_xla, "device") else xm.xla_device()
    _ = (torch.ones(2, 2, device=device) * 2).sum().item()   # force a real TPU op
    BACKEND = "tpu"
except Exception as e:
    print(f"XLA unavailable ({type(e).__name__}: {str(e)[:160]})")
    if torch.cuda.is_available():
        device, BACKEND = torch.device("cuda"), "cuda"

def sync():
    """Flush the XLA graph. No-op off TPU."""
    if BACKEND == "tpu":
        torch_xla.sync() if hasattr(torch_xla, "sync") else xm.mark_step()

print("=" * 80)
print(f"BACKEND ACTUALLY IN USE: {BACKEND.upper()}   (device={device})")
if BACKEND == "tpu":
    print(f"torch {torch.__version__} | torch_xla {torch_xla.__version__}")
print("=" * 80)

### Paths

In [ ]:
import os, json, re, gc, time, random, pickle, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd

RAW_BASE = "https://raw.githubusercontent.com/Rania-khaoudane/MSA/main/data"

def _repo_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "data" / "qa_pairs_wiki.json").exists():
            return base
    return None

ROOT = _repo_root()

def find_file(*names, subdirs=("data", "results")):
    """Locate a file: repo subdirs first, then the working directory."""
    for n in names:
        if ROOT:
            for sd in subdirs:
                p = ROOT / sd / n
                if p.exists():
                    return p
        if Path(n).exists():
            return Path(n)
    return None

def resolve(*names):
    """Local -> GitHub. Returns (json, description)."""
    p = find_file(*names, subdirs=("data",))
    if p:
        return json.loads(p.read_text(encoding="utf-8")), f"local: {p}"
    for n in names:
        try:
            url = f"{RAW_BASE}/{n}"
            with urllib.request.urlopen(url) as r:
                return json.loads(r.read().decode("utf-8")), f"github: {url}"
        except Exception:
            continue
    raise FileNotFoundError(f"none of {names} found locally or on GitHub")

OUT_DIR = (ROOT / "results") if ROOT else Path(".")
OUT_DIR.mkdir(exist_ok=True)
def out(name):
    return str(OUT_DIR / name)

print("repo root :", ROOT or "(not in the repo)")
print("output dir:", OUT_DIR.resolve())

### Config

In [ ]:
CONFIG = {
    "base_encoder": "intfloat/multilingual-e5-base",
    "reranker": "BAAI/bge-reranker-v2-m3",
    "alpha": 0.8,
    "retrieve_k": 20,
    "k_values": (1, 3, 5),
    "bootstrap_n": 1000,
    "seed": 42,
    "max_length": 512,
    "batch_size": 32,
}
CONFIG

### Load data

In [ ]:
corpus, src_c = resolve("corpus_v2.json", "corpus.json")
wiki_qa, src_q = resolve("qa_pairs_wiki.json")
print("corpus from", src_c); print("qa     from", src_q)

corpus_ids = [c["chunk_id"] for c in corpus]
corpus_texts = [c["text"] for c in corpus]
corpus_map = dict(zip(corpus_ids, corpus_texts))
known = set(corpus_ids)
qa = [q for q in wiki_qa if q["source_chunk_id"] in known]
print(f"Corpus {len(corpus)} | evaluating all {len(qa)} items")

### BM25 + hybrid scoring

In [ ]:
from rank_bm25 import BM25Okapi

DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]")

def normalize_arabic(t):
    t = DIAC.sub("", t)
    t = re.sub(r"[\u0625\u0623\u0622\u0627]", "\u0627", t)
    t = re.sub(r"\u0649", "\u064A", t); t = re.sub(r"\u0629", "\u0647", t)
    t = re.sub(r"\u0624", "\u0648", t); t = re.sub(r"\u0626", "\u064A", t)
    t = re.sub(r"\u0640+", "", t); t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def tokenize(t):
    return normalize_arabic(t).split()

bm25 = BM25Okapi([tokenize(t) for t in corpus_texts])
_bm = {}
def bm25_scores(q):
    if q not in _bm:
        _bm[q] = np.asarray(bm25.get_scores(tokenize(q)))
    return _bm[q]

def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

print("BM25 index built")

### Stage 1: retrieve, and record the DEPLOYABLE margin

In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

def batched_fixed(items, bs):
    """Yield (padded_batch, n_real); last batch padded up to bs to keep XLA shapes static."""
    for i in range(0, len(items), bs):
        ch = list(items[i:i + bs]); n = len(ch)
        if n < bs:
            ch += [ch[-1]] * (bs - n)
        yield ch, n

def mean_pool(h, mask):
    m = mask.unsqueeze(-1).to(h.dtype)
    return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)

@torch.no_grad()
def encode_texts(model, tk, texts, bs=32, max_len=512, label=""):
    """e5-style mean pooling + L2 normalisation. Verified to match
    sentence-transformers to within 3e-8 on this corpus."""
    o, done, t0 = [], 0, time.time()
    for ch, n in batched_fixed(texts, bs):
        enc = tk(ch, padding="max_length", truncation=True, max_length=max_len, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        v = mean_pool(model(**enc).last_hidden_state, enc["attention_mask"])
        v = torch.nn.functional.normalize(v, p=2, dim=1)
        sync()
        o.append(v.float().cpu().numpy()[:n]); done += n
        if done % (bs * 20) < bs:
            print(f"    {label} {done}/{len(texts)} ({time.time()-t0:.0f}s)", flush=True)
    return np.concatenate(o, 0).astype("float32")

@torch.no_grad()
def score_pairs(model, tk, pairs, bs, max_len, label=""):
    """Cross-encoder relevance score per (query, passage) pair, static shapes for XLA."""
    o, done, t0 = [], 0, time.time()
    for ch, n in batched_fixed(pairs, bs):
        enc = tk([a for a, _ in ch], [b for _, b in ch], padding="max_length",
                 truncation=True, max_length=max_len, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        logits = model(**enc).logits
        sync()
        s = logits.float().cpu().numpy()
        # Same guard as the GPU notebook: some rerankers emit a 2D per-class array.
        s = s[:, 0] if s.shape[-1] == 1 else s[:, -1]
        o.extend(s[:n].tolist()); done += n
        if done % (bs * 20) < bs:
            print(f"    {label} {done}/{len(pairs)} ({time.time()-t0:.0f}s)", flush=True)
    return np.asarray(o)

print("XLA helpers ready")

In [ ]:
print(f"Building index on {BACKEND.upper()}...")
enc_tok = AutoTokenizer.from_pretrained(CONFIG["base_encoder"])
enc_model = AutoModel.from_pretrained(CONFIG["base_encoder"]).to(device=device, dtype=torch.float32).eval()

corpus_emb = encode_texts(enc_model, enc_tok, [f"passage: {t}" for t in corpus_texts], label="corpus")
# Batch-encode every query once instead of one bi.encode() call per query.
q_emb = {f: encode_texts(enc_model, enc_tok, [f"query: {q[f]}" for q in qa], label=f)
         for f in ["msa_query", "darija_query"]}

K = CONFIG["retrieve_k"]
retrieved, ties = {}, {}
for field in ["msa_query", "darija_query"]:
    retrieved[field], n_tie = {}, 0
    for i, q in enumerate(qa):
        s = (CONFIG["alpha"] * minmax(corpus_emb @ q_emb[field][i])
             + (1 - CONFIG["alpha"]) * minmax(bm25_scores(q[field])))
        order = np.argsort(-s)[:K]
        # margin = s(top1) - s(top2): no gold label, computable at query time.
        margin = float(s[order[0]] - s[order[1]]) if len(order) > 1 else 1.0
        if margin == 0.0:
            n_tie += 1
        retrieved[field][q["id"]] = {"candidates": [corpus_ids[j] for j in order], "margin": margin}
    ties[field] = n_tie
    m = [v["margin"] for v in retrieved[field].values()]
    print(f"  {field:<14} mean top1-top2 margin = {np.mean(m):.4f}  median = {np.median(m):.4f}"
          f"  exact ties: {n_tie}")

del enc_model, corpus_emb, q_emb
gc.collect()

### Is the margin systematically lower for Darija? (the method's premise)

In [ ]:
# If dialectal queries did not have lower margins, gating on the margin would
# not be dialect-aware and the method's main claim would not hold.
rng = np.random.default_rng(CONFIG["seed"])
m_msa = np.array([retrieved["msa_query"][q["id"]]["margin"] for q in qa])
m_dar = np.array([retrieved["darija_query"][q["id"]]["margin"] for q in qa])

d = m_msa - m_dar
idx = rng.integers(0, len(d), size=(CONFIG["bootstrap_n"], len(d)))
lo, hi = np.percentile(d[idx].mean(axis=1), [2.5, 97.5])

print("=" * 78)
print("PREMISE CHECK - do dialectal queries have lower retrieval margins?")
print("=" * 78)
print(f"  MSA    mean margin  {m_msa.mean():.4f}")
print(f"  Darija mean margin  {m_dar.mean():.4f}")
print(f"  difference          {d.mean():+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]  "
      f"{'CONFIRMED' if lo > 0 else 'NOT CONFIRMED'}")
if lo <= 0:
    print("\n  NOT CONFIRMED -> margin-gating is not dialect-aware. Report this")
    print("  rather than the intended result; the gating curves below still")
    print("  describe a valid compute/accuracy tradeoff, just not a dialectal one.")

### Stage 2: rerank EVERYTHING once, so any threshold can be evaluated after

In [ ]:
print(f"\nReranking all queries once on {BACKEND.upper()} (reused for every threshold)...")
ce_tok = AutoTokenizer.from_pretrained(CONFIG["reranker"], trust_remote_code=True)
ce = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["reranker"], trust_remote_code=True).to(device=device, dtype=torch.float32).eval()

reranked = {}
for field in ["msa_query", "darija_query"]:
    qids = [q["id"] for q in qa]
    flat = [(q[field], corpus_map[c]) for q in qa for c in retrieved[field][q["id"]]["candidates"]]
    sc = score_pairs(ce, ce_tok, flat, CONFIG["batch_size"], CONFIG["max_length"], field)
    d = {}
    for i, qid in enumerate(qids):
        cands = retrieved[field][qid]["candidates"]
        s = sc[i * K:(i + 1) * K]
        d[qid] = [cands[j] for j in np.argsort(-s)]
    reranked[field] = d
    print(f"  {field} reranked")

del ce
gc.collect()

### Evaluate any gating threshold

`evaluate_gated` now takes an explicit `force` argument for the two endpoints rather than relying
on `τ = 0.0` to mean "never". With `margin <= tau`, a query whose top two candidates tie scores
`margin == 0.0` and would be routed *to* the reranker at `τ = 0`, so the "never rerank" baseline
was not guaranteed reranker-free — and that baseline is the denominator for every
`benefit_kept_%` figure below.

In [ ]:
def evaluate_gated(field, tau, force=None):
    """Rerank only queries whose margin <= tau; keep retrieval order otherwise.

    force=False -> never rerank (the true baseline, independent of ties)
    force=True  -> always rerank
    """
    hits = {k: [] for k in CONFIG["k_values"]}
    rr, flags = [], []
    for q in qa:
        info = retrieved[field][q["id"]]
        use = force if force is not None else (info["margin"] <= tau)
        ordered = reranked[field][q["id"]] if use else info["candidates"]
        flags.append(int(use))
        gold = q["source_chunk_id"]
        pos = ordered.index(gold) + 1 if gold in ordered else None
        for k in CONFIG["k_values"]:
            hits[k].append(1.0 if (pos and pos <= k) else 0.0)
        rr.append(1.0 / pos if pos else 0.0)
    return {**{f"R@{k}": np.array(v) for k, v in hits.items()},
            "MRR": np.array(rr), "reranked": np.array(flags)}

TAUS = [0.01, 0.02, 0.03, 0.05, 0.07, 0.10, 0.15, 0.20, 0.30, 0.50]

rows = []
for field in ["msa_query", "darija_query"]:
    for tau, force in [(0.0, False)] + [(t, None) for t in TAUS] + [(np.inf, True)]:
        r = evaluate_gated(field, tau, force)
        rows.append({"query": field, "tau": tau,
                     "R@1": r["R@1"].mean(), "R@5": r["R@5"].mean(), "MRR": r["MRR"].mean(),
                     "frac_reranked": r["reranked"].mean()})
curve = pd.DataFrame(rows)
curve.to_csv(out("margin_gating_curve.csv"), index=False)

print("=" * 78); print("GATING CURVE"); print("=" * 78)
for field in ["msa_query", "darija_query"]:
    print(f"\n--- {field} ---")
    print(curve[curve["query"] == field][["tau", "R@1", "R@5", "MRR", "frac_reranked"]]
          .to_string(index=False, float_format=lambda x: f"{x:.3f}"))

### The dialect-aware claim: is Darija reranked more often?

In [ ]:
print("\n" + "=" * 78)
print("IS GATING DIALECT-AWARE? (fraction of queries sent to the reranker)")
print("=" * 78)
pivot = curve.pivot(index="tau", columns="query", values="frac_reranked")
pivot["darija_minus_msa"] = pivot["darija_query"] - pivot["msa_query"]
print(pivot.to_string(float_format=lambda x: f"{x:.3f}"))
print("""
A positive 'darija_minus_msa' means dialectal queries are routed to the reranker
more often than MSA queries at the same threshold -- the gate adapts to dialect
without being told which queries are dialectal.""")

### Operating points: how much benefit is kept, at what cost

In [ ]:
print("\n" + "=" * 78); print("OPERATING POINTS (Darija)"); print("=" * 78)
dar = curve[curve["query"] == "darija_query"].set_index("tau")
never, always = dar.loc[0.0], dar.loc[np.inf]
print(f"  never rerank   R@1={never['R@1']:.3f}  ({never['frac_reranked']*100:.0f}% reranked)")
print(f"  always rerank  R@1={always['R@1']:.3f}  ({always['frac_reranked']*100:.0f}% reranked)")
full_gain = always["R@1"] - never["R@1"]
print(f"  full gain from always reranking: {full_gain:+.3f}\n")

if full_gain <= 0:
    print("  Reranking does not improve R@1 on this data, so 'benefit kept' is")
    print("  undefined -- the gating curve above is the result to report.")
else:
    op = []
    for tau in TAUS:
        row = dar.loc[tau]
        op.append({"tau": tau, "R@1": row["R@1"], "frac_reranked": row["frac_reranked"],
                   "benefit_kept_%": (row["R@1"] - never["R@1"]) / full_gain * 100,
                   "compute_saved_%": (1 - row["frac_reranked"]) * 100})
    opdf = pd.DataFrame(op)
    # tau needs more precision than the other columns: with a single shared
    # float_format of .1f, thresholds 0.01/0.02/0.03 all render as "0.0" --
    # indistinguishable in the very table you read to choose one.
    print(opdf.to_string(index=False, formatters={
        "tau": lambda x: f"{x:.3f}",
        "R@1": lambda x: f"{x:.3f}",
        "frac_reranked": lambda x: f"{x:.3f}",
        "benefit_kept_%": lambda x: f"{x:.1f}",
        "compute_saved_%": lambda x: f"{x:.1f}"}))
    opdf.to_csv(out("margin_gating_operating_points.csv"), index=False)

    good = opdf[opdf["benefit_kept_%"] >= 90]
    if len(good):
        best = good.loc[good["frac_reranked"].idxmin()]
        print(f"""
HEADLINE: at tau = {best['tau']:.3f}, {best['benefit_kept_%']:.0f}% of the reranking
benefit is retained while reranking only {best['frac_reranked']*100:.0f}% of queries
({best['compute_saved_%']:.0f}% of reranking compute saved).""")
    else:
        print("\nNo threshold retained >=90% of the benefit; report the full curve instead.")

### Effect on the dialect gap across thresholds

In [ ]:
print("\n" + "=" * 78); print("DIALECT GAP ACROSS THRESHOLDS"); print("=" * 78)
gap = curve.pivot(index="tau", columns="query", values="R@1")
gap["gap"] = gap["msa_query"] - gap["darija_query"]
gap["mean_frac_reranked"] = curve.groupby("tau")["frac_reranked"].mean()
print(gap.to_string(float_format=lambda x: f"{x:.3f}"))
gap.to_csv(out("margin_gating_gap.csv"))

print(f"\nAll outputs written under: {OUT_DIR.resolve()}")
for f in ["margin_gating_curve.csv", "margin_gating_operating_points.csv", "margin_gating_gap.csv"]:
    print("  ", f)

# The original ended with a bare `from google.colab import files`, which raises
# outside Colab and aborted the notebook on the final cell.
try:
    from google.colab import files as colab_files
    for f in ["margin_gating_curve.csv", "margin_gating_operating_points.csv", "margin_gating_gap.csv"]:
        colab_files.download(out(f))
except ImportError:
    print("(Not in Colab - files are on disk at the path above.)")